In [53]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline

In [11]:
df=pd.read_csv("D:\\AshleshaRuchika\\PGCP-AI\\Machine Learning\\Datasets\\milk.csv",index_col=0)
df

,water,protein,fat,lactose,ash
Animal,,,,,
HORSE,90.1,2.6,1.0,6.9,0.35
ORANGUTAN,88.5,1.4,3.5,6.0,0.24
MONKEY,88.4,2.2,2.7,6.4,0.18
DONKEY,90.3,1.7,1.4,6.2,0.40
HIPPO,90.4,0.6,4.5,4.4,0.10
CAMEL,87.7,3.5,3.4,4.8,0.71
BISON,86.9,4.8,1.7,5.7,0.90
BUFFALO,82.1,5.9,7.9,4.7,0.78
GUINEA PIG,81.9,7.4,7.2,2.7,0.85


In [18]:
scaler=StandardScaler().set_output(transform='pandas')
df_scaled=scaler.fit_transform(df)
dbscan=DBSCAN(eps=0.4,min_samples=2)
dbscan.fit(df_scaled)
dbscan.labels_
# silhouette_score(df_scaled,dbscan.labels_)

array([-1,  0,  0, -1, -1,  1,  1,  2, -1, -1,  2,  1, -1, -1,  1,  2, -1,
       -1, -1, -1,  3,  3, -1, -1, -1])

In [19]:
df_scaled_inlier=df_scaled.copy()
df_scaled_inlier['Cluster']=dbscan.labels_
df_scaled_inlier=df_scaled_inlier[df_scaled_inlier['Cluster']!=-1]
len(df_scaled_inlier['Cluster'].unique())

4

In [20]:
silhouette_score(df_scaled_inlier.drop('Cluster',axis=1),df_scaled_inlier['Cluster'])

0.6518937593821538

In [27]:
epsilons=np.linspace(0.01,2,100)
min_pct=[2,3,4,5]
scores=[]
for e in epsilons:
    for m in  min_pct:
        dbscan=DBSCAN(eps=e,min_samples=m)
        dbscan.fit(df_scaled)
        df_scaled_inlier=df_scaled.copy()
        df_scaled_inlier['Cluster']=dbscan.labels_
        df_scaled_inlier=df_scaled_inlier[df_scaled_inlier['Cluster']!=-1]
        if len(df_scaled_inlier['Cluster'].unique())>=2:
            sil=silhouette_score(df_scaled_inlier.drop('Cluster',axis=1),df_scaled_inlier['Cluster'])
            scores.append([e,m,sil])
df_scores=pd.DataFrame(scores,columns=['eps','min_pct','score'])
df_scores.sort_values('score',ascending=False)

,eps,min_pct,score
0,0.311515,2,0.917544
1,0.331616,2,0.903367
2,0.351717,2,0.864756
65,0.854242,3,0.657551
73,0.934646,3,0.657551
...,...,...,...
78,0.994949,2,0.434482
53,0.773838,3,0.418695
46,0.713535,3,0.418695
48,0.733636,3,0.418695


In [29]:
dbscan=DBSCAN(eps=0.432121,min_samples=3)
dbscan.fit(df_scaled)
dbscan.labels_

array([-1,  0,  0,  0, -1,  2,  2,  1, -1, -1,  1,  2,  0, -1,  2,  1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1])

In [35]:
from ipywidgets import interact
import ipywidgets as widgets

In [44]:
def dbscan_clust(e,m):
    dbscan=DBSCAN(eps=e,min_samples=m)
    dbscan.fit(df_scaled)
    return dbscan.labels_

In [ ]:
interact(dbscan_clust,e=widgets.FloatSlider(min=0.01,max=2,step=0.01,value=0.3),
                      m=widgets.IntSlider(min=2,max=5,step=1,value=2))

interactive(children=(FloatSlider(value=0.3, description='e', max=2.0, min=0.01, step=0.01), IntSlider(value=2…

<function __main__.dbscan_clust(e, m)>

In [ ]:
interact(dbscan_clust,e=widgets.FloatSlider(min=0.01,max=2,step=0.01,value=0.3),
                      m=[2,3,4,5])

interactive(children=(FloatSlider(value=0.3, description='e', max=2.0, min=0.01, step=0.01), Dropdown(descript…

<function __main__.dbscan_clust(e, m)>

NameError: name 'transf' is not defined